# Ch.9 — ML Experiment Tracking & Model Registry (Azure Supplement)

**Azure-specific extension:** This notebook adapts the main `notebook.ipynb` concepts to Azure ML infrastructure. Instead of local MLflow tracking, we explore:

- **Azure ML Workspace** connection and experiment logging
- **Azure ML HyperDrive** for distributed hyperparameter sweeps
- **Azure ML Model Registry** for versioned model storage
- **Azure Cost Tracking** for experiment budgeting

**Prerequisites:**
- Complete the main `notebook.ipynb` first (understands MLflow, experiment tracking, model registry)
- Azure subscription (Free Tier sufficient for demo — uses compute clusters for training)
- Azure ML SDK installed: `pip install azureml-core azureml-sdk`

**What you'll build:**
1. Azure ML workspace connection (credential-based or managed identity)
2. Experiment logging with Azure ML Run API
3. HyperDrive hyperparameter sweep (distributed across compute cluster)
4. Model registry workflow (register, version, download)
5. Cost tracking dashboard (per-experiment compute costs)
6. Comparison: Azure ML vs. MLflow UI

**Running example:** Same BERT fine-tuning from main notebook, but scaled to Azure compute.

In [ ]:
# TODO: Implement this cell


In [ ]:
# TODO: Implement this cell


## 1 · Log Experiment to Azure ML

Azure ML experiments work similarly to MLflow, but with tighter integration to Azure compute and storage:

**Key differences from MLflow:**
- **Experiments** are organized in the Azure ML workspace (visible in Azure Portal)
- **Runs** are automatically tracked with compute metadata (VM type, region, duration)
- **Artifacts** (models, plots) are stored in Azure Blob Storage (workspace default storage)
- **Metrics** can trigger Azure Monitor alerts (e.g., "notify if accuracy < 0.9")

**Running example:** Log a baseline BERT fine-tuning run (same as MLflow notebook).

**Cost note:** Starting a `Run` without compute is free (just metadata logging). Actual training on Azure ML compute incurs charges.

In [ ]:
# TODO: Implement this cell


## 2 · Hyperparameter Sweep with Azure ML HyperDrive

**HyperDrive** = Azure ML's distributed hyperparameter tuning service

**Key features:**
- **Sampling strategies:** Grid, random, Bayesian
- **Early termination:** Bandit policy (stop underperforming runs early)
- **Distributed execution:** Run 10+ experiments in parallel on compute cluster
- **Cost optimization:** Terminate bad runs quickly, scale down cluster after sweep

**Running example:** Same 12-run grid search from MLflow notebook:
- Learning rate: [1e-5, 3e-5, 5e-5]
- Batch size: [16, 32]
- Warmup steps: [100, 500]

**Cost estimate (for reference):**
- VM: `Standard_NC6s_v3` (1×V100, $3.06/hr)
- Duration: ~20 min per run × 12 runs = 4 hours total
- Cost: $3.06/hr × 4 hrs = **$12.24** (if runs are sequential)
- With 4-node cluster (parallel): 1 hour × $3.06 × 4 = **$12.24** (same cost, faster)

**Note:** This cell demonstrates HyperDrive *configuration* only. Actual execution requires Azure ML compute cluster.

In [ ]:
# TODO: Implement this cell


## 3 · Azure ML Model Registry

**Model Registry** = Centralized model storage with versioning and metadata

**Key differences from MLflow Model Registry:**
- Models are stored in Azure Blob Storage (workspace storage account)
- **Versioning:** Automatic (each registration creates a new version)
- **Stages:** Azure ML uses *tags* instead of stages (e.g., `stage=production`)
- **Deployment:** One-click deploy to Azure ML endpoints, ACI, AKS

**Workflow:**
1. **Register model** from run artifacts (e.g., `model.pt` from best HyperDrive run)
2. **Tag model** with metadata (e.g., `stage=staging`, `accuracy=0.94`)
3. **Download model** for inference (or deploy to Azure ML endpoint)
4. **Promote model** by updating tags (e.g., `stage=production`)

**Running example:** Register best model from HyperDrive sweep.

In [ ]:
# TODO: Implement this cell


In [ ]:
# TODO: Implement this cell


## 4 · Azure ML vs. MLflow UI Comparison

**Side-by-side feature comparison:**

| Feature | MLflow (Local) | Azure ML Studio |
|---------|----------------|------------------|
| **Experiment tracking** | ✅ Local SQLite DB | ✅ Cloud (blob storage) |
| **Model registry** | ✅ Local filesystem | ✅ Cloud (versioned) |
| **UI access** | localhost:5000 | portal.azure.com |
| **Parallel coordinates** | ✅ Built-in | ✅ HyperDrive viewer |
| **Artifact storage** | Local `mlruns/` | Azure Blob Storage |
| **Cost** | $0 (local compute) | Compute charges apply |
| **Collaboration** | Manual (shared DB) | Built-in (multi-user) |
| **Deployment** | Manual (Flask/FastAPI) | One-click (AML endpoints) |
| **Monitoring** | None | Azure Monitor integration |
| **Distributed training** | Manual (Ray/Dask) | Built-in (HyperDrive) |

**When to use MLflow:**
- Prototyping, local development
- No cloud budget
- Small team, simple workflows

**When to use Azure ML:**
- Production deployments
- Large-scale hyperparameter sweeps (>10 runs)
- Multi-user collaboration
- Need built-in monitoring and alerts

**💡 Pro tip:** Use MLflow locally, then migrate to Azure ML for production:
```python
# Local development with MLflow
mlflow.start_run()
mlflow.log_param("learning_rate", 5e-5)
mlflow.log_metric("accuracy", 0.94)
mlflow.pytorch.log_model(model, "model")

# Production deployment with Azure ML
run = experiment.start_logging()
run.log("learning_rate", 5e-5)
run.log("accuracy", 0.94)
run.upload_file("outputs/model.pt", "./model.pt")
Model.register(ws, "bert-model", model_path="outputs/model.pt")
```

## 5 · Cost Tracking for Experiments

**Cost breakdown for Azure ML experiments:**

**Compute costs (primary driver):**
- **VM instance:** Billed per second while running
- **Idle time:** No charges if cluster scales to 0 nodes
- **Storage:** ~$0.02/GB/month (minimal for model artifacts)

**Example costs (April 2026 pricing, East US):**
- `Standard_NC6s_v3` (1×V100): $3.06/hr
- `Standard_NC24s_v3` (4×V100): $12.24/hr
- `Standard_NC24ads_A100_v4` (1×A100): $5.50/hr

**Running example cost estimate:**
- **Single experiment:** 20 min × $3.06/hr = **$1.02**
- **HyperDrive sweep (12 runs, parallel):** 1 hr × $3.06 × 4 nodes = **$12.24**
- **HyperDrive sweep (12 runs, sequential):** 4 hrs × $3.06 × 1 node = **$12.24** (same cost, slower)

**Cost optimization tips:**
1. **Scale to zero:** Set `min_nodes=0` in cluster config
2. **Early termination:** Use BanditPolicy to stop bad runs
3. **Spot instances:** Save 60-80% with low-priority VMs (may be preempted)
4. **Smaller VMs:** Use `NC4as_T4_v3` ($0.53/hr) for prototyping
5. **Monitoring:** Set up budget alerts in Azure Cost Management

Below we simulate a cost tracking dashboard.

In [ ]:
# TODO: Implement this cell


## Summary & Next Steps

**What you learned:**
1. ✅ Azure ML workspace connection and experiment logging
2. ✅ HyperDrive configuration for distributed hyperparameter sweeps
3. ✅ Model registry workflow (register, version, download)
4. ✅ Cost tracking and optimization strategies
5. ✅ Azure ML vs. MLflow comparison

**Key takeaways:**
- **Local MLflow** for prototyping (free, fast iteration)
- **Azure ML** for production (scalable, collaborative, monitored)
- **Cost optimization:** Scale to zero + early termination = 40-60% savings
- **Model registry** enables reproducibility (version + metadata + artifacts)

**Next steps:**
1. **Deploy model to Azure ML endpoint** (see Ch.10 for inference servers)
2. **Set up monitoring** (data drift, prediction drift, performance)
3. **A/B test models** (v1 vs. v2 in production)
4. **Automate retraining** (Azure ML pipelines for CI/CD)

**Resources:**
- [Azure ML Documentation](https://docs.microsoft.com/en-us/azure/machine-learning/)
- [HyperDrive Guide](https://docs.microsoft.com/en-us/azure/machine-learning/how-to-tune-hyperparameters)
- [Model Registry Best Practices](https://docs.microsoft.com/en-us/azure/machine-learning/how-to-manage-models)
- [Azure ML Pricing Calculator](https://azure.microsoft.com/en-us/pricing/calculator/)